# DriftBeacon GPU training on Google Colab

This notebook trains the prepared XGBoost candidate. It uses only the supplied train and validation data. The final test data stays with Adarsh and is evaluated after the model is returned.

Before running: choose **Runtime > Change runtime type > T4 GPU** (or another available NVIDIA GPU). Colab GPU availability is dynamic, so CPU mode is also supported.

In [ ]:
!nvidia-smi
import os, sys, json, shutil, zipfile
print(sys.version)
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES'))

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(zip_name) as z:
    bad = [n for n in z.namelist() if n.startswith('/') or '..' in n.replace('\\', '/').split('/')]
    if bad: raise ValueError(f'Unsafe ZIP paths: {bad[:3]}')
    z.extractall('/content')
ROOT = '/content/driftbeacon'
assert os.path.exists(ROOT + '/model/train.py'), ROOT
%cd /content/driftbeacon
print('Prepared package:', ROOT)

In [ ]:
!pip install -q -r requirements-colab.txt
!python -m model.verify --training-only
!python -m unittest discover -s tests -q

## Train the GPU candidate

This command uses all prepared training rows (`--max-train-rows 0`). It compares the median baseline, Ridge, and XGBoost on the frozen validation vehicles. The script checks that XGBoost actually selected CUDA and saves the winning model.

In [ ]:
!python -m model.train --algorithm xgboost --device cuda --max-train-rows 0 --jobs 4 --output artifacts/models/gpu_candidate

In [ ]:
# Calibrate the detector for this exact model using validation only.
!python -m model.evaluate --model artifacts/models/gpu_candidate --calibrate
!python -m model.replay --model artifacts/models/gpu_candidate --output reports/replays/gpu_candidate_replay.json

In [ ]:
# Download a single artifact bundle to return to Adarsh.
from google.colab import files
archive = shutil.make_archive('/content/driftbeacon-gpu-candidate', 'zip', '/content/driftbeacon/artifacts/models/gpu_candidate')
print(archive)
files.download(archive)

Return the downloaded ZIP together with these two report files from the Colab session:
- `reports/gpu_candidate_training.json`
- `reports/gpu_candidate_detector_validation.json`

Also report the GPU name from `nvidia-smi`, the Colab runtime Python version, training duration, and whether the full `--max-train-rows 0` command completed. Do not run final test evaluation in Colab because the test data is intentionally absent.